# NHL Draft Analysis - Determining the Performance Score

## Setup

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../data/nhl_drafts_clean.db")
print("Connected!")

Connected!


In [2]:
# Load full dataset
df = pd.read_sql_query("""
    SELECT draft_year, round, pick_number, drafted_by, player_name, 
           position, games_played, goals, assists, points, penalty_minutes
    FROM nhl_draft_picks
    ORDER BY draft_year, pick_number
""", conn)

df['games_played'] = df['games_played'].fillna(0)
df['points'] = df['points'].fillna(0)

print(f"Loaded {len(df)} players")
df.head()

Loaded 2766 players


,draft_year,round,pick_number,drafted_by,player_name,position,games_played,goals,assists,points,penalty_minutes
0,2005,1,1,Pittsburgh,Sidney Crosby,C,1408.0,652.0,1094.0,1746.0,892.0
1,2005,1,2,Anaheim,Bobby Ryan,R,866.0,261.0,308.0,569.0,470.0
2,2005,1,3,Carolina,Jack Johnson,D,1228.0,77.0,265.0,342.0,639.0
3,2005,1,4,Minnesota,Benoit Pouliot,L,625.0,130.0,133.0,263.0,371.0
4,2005,1,5,Montreal,Carey Price,G,712.0,0.0,13.0,13.0,51.0


## The Formula

The goal of this analysis is to measure team draft effectiveness. We do 
this by comparing what each team actually did in a draft against what the 
best possible outcome would have looked like in hindsight — a hindsight 
ranking of every player in that draft by their actual NHL career performance.

For each pick, we calculate a plus/minus score:

**plus/minus = actual pick number - hindsight rank**

A positive score means the team got more value than expected at that pick. 
A negative score means they got less. Summing these scores across all picks 
and all drafts gives us a single number per team that captures their overall 
drafting effectiveness.

To build the hindsight ranking we need a single metric that captures NHL 
career effectiveness. The two statistics available to us are career points 
and career games played. Points alone would systematically undervalue 
defensemen and reliable depth players. Games played alone would fail to 
distinguish franchise cornerstones from journeymen. The formula combines both:

**Performance Score = games_played + (points * X)**

The coefficient X controls how much weight points carry relative to games 
played. We derive X from the data itself — the value at which both components 
contribute equally on average:

**avg_games_played = avg_points * X**

**X = avg_games_played / avg_points**

Note: Goalies are excluded from the hindsight ranking. Because goalies 
accumulate minimal points regardless of their quality, they would rank near 
the bottom of every hindsight draft, producing large negative plus/minus 
scores for teams that invested early picks in goalies — even when those 
were correct decisions. Excluding them means goalie picks do not count 
toward a team's score, which is acknowledged as a limitation but is 
significantly fairer than the alternative.

In [3]:
nhl_players = df[(df['games_played'] > 0) & (df['points'] > 0)]
avg_gp = nhl_players['games_played'].mean()
avg_pts = nhl_players['points'].mean()
natural_X = round(avg_gp / avg_pts, 2)

print(f"Average GP:                {avg_gp:.1f}")
print(f"Average Points:            {avg_pts:.1f}")
print(f"Natural equal-weight X:    {natural_X}")

Average GP:                367.4
Average Points:            160.2
Natural equal-weight X:    2.29


With a natural equal-weight X of 2.29, both components contribute equally 
to the performance score on average. This is our starting point — derived 
directly from the data with no assumptions about whether longevity or 
scoring is more valuable.

Before committing to this value we run two checks:
1. **Positional Fairness** — does X=2.29 significantly underrepresent 
   defensemen relative to what the actual drafts would suggest?
2. **Sensitivity** — if we deviated one step in either direction, would 
   it meaningfully change the rankings?

If both checks pass, X=2.29 is confirmed as the final coefficient.

## Two Checks Before We Commit

Throughout both checks we evaluate the hindsight top 30 — representing 
roughly one pick per team per draft, the most scrutinized tier of any 
draft class and the range where teams invest the most scouting resources.

### Check 1 - Positional Fairness

We compare the positional breakdown of the hindsight top 30 at three values 
of X against the actual top 30 picks across all 13 drafts. The three values 
tested are X=1.29, X=2.29, and X=3.29 — one step below, the center, and 
one step above, where the gap is derived as 1.0.

If X=2.29 significantly underrepresents defensemen relative to the actual 
baseline — we define significant as more than 5 percentage points, or roughly 
2 extra forwards per draft class — we would consider moving to a lower value 
of X to improve positional fairness at the cost of equal weighting.

In [4]:
def show_position_breakdown(df, x_values, top_n=30):
    df = df.copy()
    df = df[df['position'] != 'G']

    # Baseline — actual top N picks positional breakdown
    top_n_actual = df[df['pick_number'] <= top_n].copy()
    top_n_actual['position_group'] = top_n_actual['position'].apply(
        lambda x: 'D' if x == 'D' else 'F'
    )
    baseline = top_n_actual.groupby('position_group').size().reset_index(name='count')
    baseline['percentage'] = (baseline['count'] / len(top_n_actual) * 100).round(1)
    baseline = baseline.set_index('position_group')['percentage']

    print(f"Positional breakdown of hindsight top {top_n} vs actual top {top_n} picks\n")
    print(f"{'X Value':<12} {'F %':<10} {'D %':<10} {'F % actual':<15} {'D % actual':<15} {'F diff':<10} {'D diff'}")
    print("-" * 85)

    for X in x_values:
        df['performance_score'] = df['games_played'] + (df['points'] * X)
        df['hindsight_rank'] = df.groupby('draft_year')['performance_score'].rank(ascending=False, method='min')

        top_n_df = df[df['hindsight_rank'] <= top_n].copy()
        top_n_df['position_group'] = top_n_df['position'].apply(
            lambda x: 'D' if x == 'D' else 'F'
        )

        breakdown = top_n_df.groupby('position_group').size().reset_index(name='count')
        total = breakdown['count'].sum()
        breakdown['percentage'] = (breakdown['count'] / total * 100).round(1)
        breakdown = breakdown.set_index('position_group')['percentage']

        f_pct = breakdown.get('F', 0)
        d_pct = breakdown.get('D', 0)
        f_actual = baseline.get('F', 0)
        d_actual = baseline.get('D', 0)
        f_diff = round(f_pct - f_actual, 1)
        d_diff = round(d_pct - d_actual, 1)

        print(f"X={X:<10} {f_pct:<10} {d_pct:<10} {f_actual:<15} {d_actual:<15} {f_diff:<10} {d_diff}")

`show_position_breakdown` takes the full dataset and a list of X values 
and compares the positional breakdown of the hindsight top 30 against the 
actual top 30 picks across all 13 drafts. It reports the forward and 
defenseman percentages at each X value alongside the actual baseline and 
the difference between them.

In [5]:
def show_rank_movement(df, year, x_values, top_n=30):
    df = df.copy()
    df = df[df['position'] != 'G']
    
    rank_data = {}
    for X in x_values:
        df['performance_score'] = df['games_played'] + (df['points'] * X)
        df[f'rank_X{X}'] = df.groupby('draft_year')['performance_score'].rank(ascending=False, method='min')
        rank_data[X] = df[df['draft_year'] == year][['player_name', 'position', 'pick_number', f'rank_X{X}']].copy()
    
    merged = rank_data[x_values[0]][['player_name', 'position', 'pick_number', f'rank_X{x_values[0]}']].copy()
    for X in x_values[1:]:
        merged = merged.merge(rank_data[X][['player_name', f'rank_X{X}']], on='player_name')
    
    rank_cols = [f'rank_X{X}' for X in x_values]
    merged = merged[merged[rank_cols].min(axis=1) <= top_n]
    
    # Calculate difference from center value (X=2.29)
    center_col = f'rank_X{x_values[1]}'
    for col in [f'rank_X{x_values[0]}', f'rank_X{x_values[2]}']:
        merged[f'diff_{col}'] = (merged[col] - merged[center_col]).abs()
    
    merged['avg_diff_from_center'] = merged[[f'diff_rank_X{x_values[0]}', 
                                              f'diff_rank_X{x_values[2]}']].mean(axis=1).round(1)
    merged['avg_rank'] = merged[rank_cols].mean(axis=1)
    merged = merged.sort_values('avg_rank')
    
    merged.columns = ['player', 'pos', 'actual_pick'] + \
                     [f'X={X}' for X in x_values] + \
                     [f'diff_from_center_low', 'diff_from_center_high', 
                      'avg_diff_from_center', 'avg_rank']
    merged = merged.reset_index(drop=True)
    
    avg_overall = merged['avg_diff_from_center'].mean().round(2)
    print(f"{year} Draft — Rank Movement (center = X=2.29)")
    print(merged[['player', 'pos', 'actual_pick', 
                  f'X={x_values[0]}', f'X={x_values[1]}', f'X={x_values[2]}',
                  'avg_diff_from_center']].to_string())
    print(f"\nAverage rank difference from X=2.29: {avg_overall} spots")

`show_rank_movement` takes the full dataset, a draft year, and a list of 
X values and shows how each player in the top 30 ranks at each coefficient, 
with X=2.29 as the anchor. The `avg_diff_from_center` column reports the 
average number of spots each player moves when shifting one step in either 
direction from the center value.

In [6]:
gap = 1.0
x_values = [round(natural_X - gap, 2), natural_X, round(natural_X + gap, 2)]
print(f"Test values: {x_values}")

Test values: [1.29, 2.29, 3.29]


In [7]:
show_position_breakdown(df, x_values)

Positional breakdown of hindsight top 30 vs actual top 30 picks

X Value      F %        D %        F % actual      D % actual      F diff     D diff
-------------------------------------------------------------------------------------
X=1.29       69.7       30.3       66.8            33.2            2.9        -2.9
X=2.29       70.8       29.2       66.8            33.2            4.0        -4.0
X=3.29       71.5       28.5       66.8            33.2            4.7        -4.7


The forward over-representation at X=2.29 is 4.0 percentage points — below 
our threshold of 5 percentage points. Across all 13 drafts this equates to 
roughly 16 additional forwards in the hindsight top 30, or just over one 
extra forward per draft class. 

**Check 1 passed.** X=2.29 does not introduce unacceptable positional bias.

### Check 2 — Sensitivity

We now look at how much the rankings change when we move one step in either 
direction from X=2.29. We use three benchmark drafts — 2005, 2007, and 2015 
— chosen because they contain well known players whose careers are easy to 
evaluate intuitively.

We measure sensitivity as the average rank difference from X=2.29 across 
all players in the top 30. If the average difference is small, the choice 
of X is not consequential and X=2.29 is confirmed.

In [8]:
show_rank_movement(df, 2005, x_values)
show_rank_movement(df, 2007, x_values)
show_rank_movement(df, 2015, x_values)

2005 Draft — Rank Movement (center = X=2.29)
                   player pos  actual_pick  X=1.29  X=2.29  X=3.29  avg_diff_from_center
0           Sidney Crosby   C            1     1.0     1.0     1.0                   0.0
1            Anze Kopitar   C           11     2.0     2.0     2.0                   0.0
2       Kristopher Letang   D           62     3.0     3.0     4.0                   0.5
3            Paul Stastny   C           44     4.0     4.0     3.0                   0.5
4              T.J. Oshie   R           24     6.0     5.0     5.0                   0.5
5            Keith Yandle   D          105     5.0     6.0     6.0                   0.5
6         Andrew Cogliano   C           25     7.0     7.0     7.0                   0.0
7     Marc-Edouard Vlasic   D           35     8.0     8.0    11.0                   1.5
8              Bobby Ryan   R            2    11.0     9.0     8.0                   1.5
9        Patric Hornqvist   R          230    10.0    11.0    10.

Across the three benchmark drafts, the average rank difference from X=2.29 
when moving one full gap unit in either direction was 0.75, 0.50, and 0.80 
spots respectively. Given that a gap of 1.0 represents a 44% deviation from 
the equal-weight center, these differences are remarkably small. The most 
sensitive player across all three drafts was Kirill Kaprizov in 2015 with 
an average difference of 4.0 spots — an outlier explained by his exceptional 
scoring rate relative to his games played.

**Check 2 passed.** Even substantial deviations from X=2.29 produce 
negligible changes in the rankings, confirming the methodology is robust.

## The Decision

Both checks passed. X=2.29 is confirmed as the final coefficient.

This value is derived directly from the data as the natural equal-weight 
point between games played and points. It makes no philosophical assumptions 
about whether longevity or scoring is more valuable. The positional fairness 
check confirmed it does not significantly underrepresent defensemen. The 
sensitivity check confirmed that deviating from it would not meaningfully 
change the conclusions.

The final performance score formula is:

**Performance Score = games_played + (points * 2.29)**

This formula will be used to build the hindsight ranking for every draft 
from 2005 to 2017, against which each team's actual selections will be 
compared to produce the final draft efficiency scores.

## Applying the Formula

With X=2.29 confirmed, we now apply the formula to the full dataset. 
For each draft year we calculate the performance score for every player, 
build the hindsight ranking, and compare it against the actual pick number 
to produce a plus/minus score.

A positive plus/minus means the player outperformed their draft position — 
they were picked later than their hindsight rank suggests they should have been. 
A negative plus/minus means they underperformed — they were picked earlier 
than their hindsight rank suggests they deserved.

In [9]:
FINAL_X = 2.29

df_scores = df[df['position'] != 'G'].copy()

# Calculate performance score
df_scores['performance_score'] = (
    df_scores['games_played'] + (df_scores['points'] * FINAL_X)
).round(2)

# Build hindsight ranking within each draft year
df_scores['hindsight_rank'] = df_scores.groupby('draft_year')['performance_score'].rank(
    ascending=False, method='min'
).astype(int)

# Calculate plus/minus
df_scores['plus_minus'] = df_scores['pick_number'] - df_scores['hindsight_rank']

print(f"Formula applied to {len(df_scores)} players")
df_scores[['draft_year', 'pick_number', 'drafted_by', 'player_name', 
           'position', 'games_played', 'points', 
           'performance_score', 'hindsight_rank', 'plus_minus']].head(20)

Formula applied to 2484 players


,draft_year,pick_number,drafted_by,player_name,position,games_played,points,performance_score,hindsight_rank,plus_minus
0,2005,1,Pittsburgh,Sidney Crosby,C,1408.0,1746.0,5406.34,1,0
1,2005,2,Anaheim,Bobby Ryan,R,866.0,569.0,2169.01,9,-7
2,2005,3,Carolina,Jack Johnson,D,1228.0,342.0,2011.18,12,-9
3,2005,4,Minnesota,Benoit Pouliot,L,625.0,263.0,1227.27,21,-17
5,2005,6,Columbus,Gilbert Brule,C,299.0,95.0,516.55,39,-33
6,2005,7,Chicago,Jack Skille,R,368.0,84.0,560.36,36,-29
7,2005,8,San Jose,Devin Setoguchi,R,516.0,261.0,1113.69,24,-16
8,2005,9,Ottawa,Brian Lee,D,209.0,36.0,291.44,45,-36
9,2005,10,Vancouver,Luc Bourdon,D,36.0,2.0,40.58,72,-62
10,2005,11,Los Angeles,Anze Kopitar,C,1503.0,1305.0,4491.45,2,9


A quick sanity check — the top picks from each draft should have a negative 
plus/minus since they were picked first but may not have been the best 
player in hindsight. And late round steals should have a large positive 
plus/minus.

In [10]:
# Best plus/minus -- late round steals
print("Top 10 largest plus/minus (biggest steals):")
steals = df_scores.nlargest(10, 'plus_minus')[
    ['draft_year', 'pick_number', 'drafted_by', 'player_name', 'hindsight_rank', 'plus_minus']
]
print(steals.to_string())

# Worst plus/minus -- early round misses
print("\nTop 10 smallest plus/minus (biggest misses):")
misses = df_scores.nsmallest(10, 'plus_minus')[
    ['draft_year', 'pick_number', 'drafted_by', 'player_name', 'hindsight_rank', 'plus_minus']
]
print(misses.to_string())

Top 10 largest plus/minus (biggest steals):
      draft_year  pick_number drafted_by       player_name  hindsight_rank  plus_minus
229         2005          230  Nashville  Patric Hornqvist              11         219
215         2005          216    Toronto    Anton Stralman              15         201
1491        2011          208  Tampa Bay      Ondrej Palat              14         194
1911        2013          206    Florida  MacKenzie Weegar              28         178
440         2006          211     Ottawa       Erik Condra              34         177
643         2007          201   San Jose      Justin Braun              25         176
221         2005          222   Colorado     Kyle Cumiskey              49         173
2123        2014          207   Montreal        Jake Evans              35         172
199         2005          200   Montreal  Sergei Kostitsyn              29         171
1252        2010          178     Ottawa        Mark Stone               8         170

From this we see that the largest positive scores exceed +200 while the largest negative scores 
are under -100. This asymmetry is worth investigating further.

In [11]:
pivot = df_scores.groupby('draft_year').agg(
    best_steal_score=('plus_minus', 'max'),
    best_steal_player=('player_name', lambda x: x[df_scores.loc[x.index, 'plus_minus'].idxmax()]),
    worst_bust_score=('plus_minus', 'min'),
    worst_bust_player=('player_name', lambda x: x[df_scores.loc[x.index, 'plus_minus'].idxmin()])
).reset_index()

print(pivot.to_string())

    draft_year  best_steal_score best_steal_player  worst_bust_score  worst_bust_player
0         2005               219  Patric Hornqvist               -88     Marek Zagrapan
1         2006               177       Erik Condra               -60        Mark Mitera
2         2007               176      Justin Braun               -77  Alexei Cherepanov
3         2008               160      Jason Demers               -85         Kyle Beach
4         2009               162          Nic Dowd               -93      Scott Glennie
5         2010               170        Mark Stone               -64        Joey Hishon
6         2011               194      Ondrej Palat               -95       Mark McNeill
7         2012               152      Jaycob Megna               -76   Griffin Reinhart
8         2013               178  MacKenzie Weegar               -71       Samuel Morin
9         2014               172        Jake Evans               -69   Conner Bleackley
10        2015               159

The plus/minus scores reveal a notable asymmetry. The largest positive 
scores exceed +200, while the largest negative scores are capped at 
roughly -100. This asymmetry is structural — pick numbers extend into 
the 200s, but the floor of the hindsight ranking is determined by the 
rank of the first player with a zero performance score, which is shared 
by all players who never reached the NHL. In the 2005 draft, 107 players 
never played an NHL game, all sharing hindsight rank 101 — meaning the 
worst possible plus/minus for a bust in that draft is -100. Meanwhile 
a player picked 230th who ranks 11th in hindsight produces a score of +219.

This means the metric rewards late round steals more than it penalizes 
early round busts. There are two ways to interpret this:

The optimistic view is that the asymmetry reflects reality. Finding a 
gem at pick 220 is genuinely more impressive than missing on pick 5 — 
late round steals are rarer and harder to find, so rewarding them more 
heavily is arguably correct.

The cautious view is that a team that drafts conservatively — making 
safe picks early and avoiding big swings late — may be systematically 
underrated compared to a team that gets lucky with one or two late round 
outliers. This is worth keeping in mind when interpreting the final team 
rankings in the analysis.

This asymmetry is acknowledged as a known limitation of the methodology.

## Franchise Consolidation

The Atlanta Thrashers relocated to become the Winnipeg Jets in 2011, 
and the Phoenix Coyotes were rebranded as the Arizona Coyotes in 2014. 
Since the front offices moved with the franchises, their draft histories 
are combined under the current franchise name.

In [12]:
# Franchise consolidation
print("Unique teams before consolidation:")
print(sorted(df_scores['drafted_by'].unique()))
print(f"\nTotal teams: {df_scores['drafted_by'].nunique()}")

franchise_map = {
    'Atlanta': 'Winnipeg',
    'Phoenix': 'Arizona'
}

df_scores['drafted_by'] = df_scores['drafted_by'].replace(franchise_map)

print("\nUnique teams after consolidation:")
print(sorted(df_scores['drafted_by'].unique()))
print(f"\nTotal teams: {df_scores['drafted_by'].nunique()}")

Unique teams before consolidation:
['Anaheim', 'Arizona', 'Atlanta', 'Boston', 'Buffalo', 'Calgary', 'Carolina', 'Chicago', 'Colorado', 'Columbus', 'Dallas', 'Detroit', 'Edmonton', 'Florida', 'Los Angeles', 'Minnesota', 'Montreal', 'NY Islanders', 'NY Rangers', 'Nashville', 'New Jersey', 'Ottawa', 'Philadelphia', 'Phoenix', 'Pittsburgh', 'San Jose', 'St. Louis', 'Tampa Bay', 'Toronto', 'Vancouver', 'Vegas', 'Washington', 'Winnipeg']

Total teams: 33

Unique teams after consolidation:
['Anaheim', 'Arizona', 'Boston', 'Buffalo', 'Calgary', 'Carolina', 'Chicago', 'Colorado', 'Columbus', 'Dallas', 'Detroit', 'Edmonton', 'Florida', 'Los Angeles', 'Minnesota', 'Montreal', 'NY Islanders', 'NY Rangers', 'Nashville', 'New Jersey', 'Ottawa', 'Philadelphia', 'Pittsburgh', 'San Jose', 'St. Louis', 'Tampa Bay', 'Toronto', 'Vancouver', 'Vegas', 'Washington', 'Winnipeg']

Total teams: 31


In [13]:
# Output to csv
output_cols = ['draft_year', 'round', 'pick_number', 'drafted_by', 'player_name',
               'position', 'games_played', 'points', 'performance_score', 
               'hindsight_rank', 'plus_minus']

df_scores[output_cols].to_csv('../data/outputs/draft_scores.csv', index=False)
print(f"draft_scores.csv saved with {len(df_scores)} rows")

draft_scores.csv saved with 2484 rows


## End of Analysis

In [14]:
conn.close()
print("Connection closed.")

Connection closed.


The formula has been applied to all 13 drafts from 2005 to 2017. 
The output has been saved to data/outputs/draft_scores.csv and will 
serve as the foundation for both the analysis notebook and Tableau dashboard.